# Breadth first search exercise

Load code and data for countries, cities, and roads, which form a symmetric weighted graph.

Search for a path between cities **a** and **b** that goes through the lowest number of nodes.

See also https://en.wikipedia.org/wiki/Breadth-first_search .


In [1]:
import os
#print(os.getcwd())

from roadbox import Country, City, Road
from roadbox import show_map, show_path

In [2]:
with open(f"countries.py", 'r', encoding="utf8") as file:
    countries = eval(file.read())

with open(f"cities.py", 'r', encoding="utf8") as file:
    cities = eval(file.read())

with open(f"roads.py", 'r', encoding="utf8") as file:
    roads = eval(file.read())

with open(f"example_paths.py", 'r', encoding="utf8") as file:
    example_paths = eval(file.read())

In [3]:
def create_map(roads):
    '''creates an adjacency map'''
    adjacency_map = {}
    for r in roads:
        if not r.a in adjacency_map:
            adjacency_map[r.a] = {}
        adjacency_map[r.a][r.b] = r.distance
        if not r.b in adjacency_map:
            adjacency_map[r.b] = {}
        adjacency_map[r.b][r.a] = r.distance
    return adjacency_map

adjacency_map = create_map(roads)

# Dijkstra's shortest path algorithm

We search for a path from city **a** to city **b**.

Because breadth-first expands the search frontier level by level, it finds the shortest path in an unweighted graph but not in a weighted graph. In a weighted graph, a long path through only a few nodes will beat a short path through many nodes.

So, instead of expanding the search frontier by distance counted in nodes regardless of distance, Dijkstra's algorithm expands the frontier by only adding neighbors of cities that are at minimal distance from the origin, and processes the cities in the frontier in their order of distance from the origin.

For this, we replace the **level** of breadth-first by a data structure **frontier** which is continuously growing and shrinking. It is efficiently managed as a priority queue by Python's heapq module.

As with breadth-first, each city has a unique path. For each city **x** we process, we record the predecessor that led to **x**, which by virtue of the priority list must be on the shortest path to **x**. We construct the shortest path from **a** to **b** from the predecessors.

In [ ]:
# This implementation uses a priority queue based on a heap structure
# further down you will also find an implementation that continuously 
# sorts the frontier.

import heapq # for the priority queue

def dijkstra(a, b, adjacency_map):
    '''finds the shortest path with Dijkstra's algorithm'''

    taboo = [] 
    predecessors = {} # to reconstruct the path

    # The frontier is a list filled with 3-tuples: 
    # (distance, city, predecessor on path to city).
    # Distance is the first element of the tuplet because the priority 
    # list needs that for sorting. 

    # The first element of frontier is a distance 0 from a and no predecessor.
    frontier = [(0, a, None)] 

    # if len(frontier)==0 then there is no path from a to b
    while len(frontier) > 0: 

        # Since frontier is a list of 3-tuples, we can use a comma separated list
        # to assign three variables to the three elements of each tuple. This is 
        # more readable than doing current = heapq.heappop(frontier) and then  
        # working with current[0], current[1], current[2]
        current_distance, current_city, current_predecessor = heapq.heappop(frontier)

        # If a city x can be reached by multiples paths/distances, x will enter
        # the frontier multiple times and will pop multiple times. 
        #
        # The priority list guarantees that heapq.heappop() pops the tuple of x
        # with the shortest distance to the origin first. 
        #
        # Any neighbors of x that are then added to the frontier will have their
        # distance to the origin calculated based on this shortest distance.
        #
        # During processing, x is added to the taboo list, to avoid processing it 
        # again with a longer distance.

        if current_city not in taboo: 

            # Since current_city is not in taboo, this is the first time we look at
            # it. Because it had the highest priority, current_path is the SHORTEST
            # distance between the origin and current_city and current_predecessor 
            # is the predecessor on the shortest path.

            predecessors[current_city] = current_predecessor

            taboo.append(current_city) 

            if b == current_city:            
                path = [b]
                predecessor = predecessors[b]
                while predecessor: # terminates when predecessor is None
                    path.append(predecessor)
                    predecessor = predecessors[predecessor]
                return path[::-1] # current_path is the path from a to b
            
            for neighbor, neighbor_distance in adjacency_map[current_city].items():
                if not neighbor in taboo:
                    new_distance = current_distance + neighbor_distance
                    # heapq.heappush makes sure that when adding an element to
                    # frontier, the heap property is preserved.
                    heapq.heappush(frontier, 
                                   (new_distance, neighbor, current_city))

    return []

In [9]:
path = dijkstra(adjacency_map, 41, 6)

show_path(cities, path, adjacency_map)

TypeError: cannot use 'dict' as a dict key (unhashable type: 'dict')

The red path above is the path found with breadth first. It is much shorter. 